In [1]:
!pip install -q pixeltable-pgserver psycopg2-binary sqlalchemy pandas kagglehub

import os
import json
import tarfile
import warnings
import statistics
import urllib.request

import pandas as pd
from sqlalchemy import create_engine, text

try:
    import pgserver
except ImportError:
    import pixeltable_pgserver as pgserver

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 90)

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri().replace('postgresql://', 'postgresql+psycopg2://'), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

def sql(query):
    return pd.read_sql(text(query), engine)

def run(query):
    with engine.begin() as conn:
        conn.execute(text(query))

def explain(query):
    with engine.begin() as conn:
        lines = [r[0] for r in conn.execute(text('EXPLAIN (ANALYZE, BUFFERS) ' + query))]
    print('\n'.join(lines))

def exec_ms(query, n=5, setup=None):
    times = []
    for _ in range(n):
        with engine.begin() as conn:
            if setup:
                conn.execute(text(setup))
            plan = conn.execute(text('EXPLAIN (ANALYZE, FORMAT JSON) ' + query)).scalar_one()
        if isinstance(plan, str):
            plan = json.loads(plan)
        times.append(plan[0]['Execution Time'])
    return round(statistics.median(times), 2)

PostgreSQL 16.15 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit


## 1. Setup і typed-схема Olist

In [2]:
FILES = [
    'olist_customers_dataset.csv',
    'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv',
    'olist_products_dataset.csv',
    'olist_sellers_dataset.csv',
]
DATA_DIR = 'data/olist'
PARTS = [f'data/olist.tar.xz.part{i:02d}' for i in range(5)]
PARTS_URL = 'https://raw.githubusercontent.com/qEmrys/goit-rdb-hw-05/main/'

def have_csv(folder):
    return all(os.path.exists(os.path.join(folder, f)) for f in FILES)

def extract_parts():
    os.makedirs(DATA_DIR, exist_ok=True)
    for p in PARTS:
        if not os.path.exists(p):
            urllib.request.urlretrieve(PARTS_URL + p, p)
    with open('/tmp/olist.tar.xz', 'wb') as out:
        for p in PARTS:
            with open(p, 'rb') as f:
                out.write(f.read())
    with tarfile.open('/tmp/olist.tar.xz', 'r:xz') as tar:
        tar.extractall(DATA_DIR, filter='data')

if have_csv(DATA_DIR):
    dataset_dir = DATA_DIR
elif os.path.exists(PARTS[0]):
    extract_parts()
    dataset_dir = DATA_DIR
else:
    try:
        import kagglehub
        dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
    except Exception as e:
        print('KaggleHub не спрацював:', type(e).__name__)
        os.makedirs('data', exist_ok=True)
        extract_parts()
        dataset_dir = DATA_DIR

print('Папка з CSV:', dataset_dir)

Папка з CSV: data/olist


In [3]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

raw_counts = {}
for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    raw_counts[table_name] = len(df)
    print(f'{table_name}: {df.shape[0]} рядків, {df.shape[1]} колонок')

olist_customers_raw: 99441 рядків, 5 колонок


olist_orders_raw: 99441 рядків, 8 колонок


olist_order_items_raw: 112650 рядків, 7 колонок


olist_products_raw: 32951 рядків, 9 колонок
olist_sellers_raw: 3095 рядків, 4 колонок


In [4]:
run("""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;
DROP TABLE IF EXISTS hw5_referrals CASCADE;
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders CASCADE;
DROP TABLE IF EXISTS olist_customers CASCADE;
DROP TABLE IF EXISTS olist_products CASCADE;
DROP TABLE IF EXISTS olist_sellers CASCADE;

CREATE TABLE olist_customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status TEXT NOT NULL,
    order_purchase_timestamp TIMESTAMP NOT NULL,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city TEXT,
    seller_state CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id INTEGER NOT NULL,
    product_id TEXT REFERENCES olist_products(product_id),
    seller_id TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
""")
print('Typed-таблиці створено')

Typed-таблиці створено


In [5]:
run("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

ANALYZE olist_customers;
ANALYZE olist_orders;
ANALYZE olist_order_items;
ANALYZE olist_products;
ANALYZE olist_sellers;
""")
print('Дані перенесено у typed-таблиці')

Дані перенесено у typed-таблиці


In [6]:
smoke = sql("""
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
""")
raw_map = {
    'customers': 'olist_customers_raw', 'orders': 'olist_orders_raw',
    'order_items': 'olist_order_items_raw', 'products': 'olist_products_raw',
    'sellers': 'olist_sellers_raw',
}
smoke['n_raw'] = smoke['table_name'].map(lambda x: raw_counts[raw_map[x]])
smoke['різниця'] = smoke['n_raw'] - smoke['n_rows']
smoke

,table_name,n_rows,n_raw,різниця
0,customers,99441,99441,0
1,order_items,112650,112650,0
2,orders,99441,99441,0
3,products,32951,32951,0
4,sellers,3095,3095,0


Кількість рядків у typed-таблицях збігається з raw для всіх 5 таблиць, різниця 0. Тобто всі рядки пройшли типи, FK і CHECK, нічого не загубилось.

## 2. CTE-chain pipeline з ≥5 stages

In [7]:
FEATURES_CTE = """
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
"""

FEATURES_SELECT = """
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
    AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
"""

PIPELINE_TOP20 = FEATURES_CTE + FEATURES_SELECT + """
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20
"""

sql(PIPELINE_TOP20)

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0,pcs,4681.78


In [8]:
sql(FEATURES_CTE + """
, features AS (
""" + FEATURES_SELECT + """
)
SELECT
    COUNT(*)::BIGINT AS n_customers,
    COUNT(*) FILTER (WHERE frequency > 1)::BIGINT AS n_freq_gt_1,
    COUNT(*) FILTER (WHERE cancellation_rate > 0)::BIGINT AS n_with_cancel,
    COUNT(*) FILTER (WHERE preferred_category = '(none)')::BIGINT AS n_no_category,
    COUNT(DISTINCT cohort_quarter)::BIGINT AS n_cohorts,
    MIN(recency_days) AS min_recency,
    MAX(recency_days) AS max_recency
FROM features
""")

,n_customers,n_freq_gt_1,n_with_cancel,n_no_category,n_cohorts,min_recency,max_recency
0,99441,0,625,775,10,1,774


In [9]:
sql("""
SELECT
    COUNT(*)::BIGINT AS n_customer_id,
    COUNT(DISTINCT customer_unique_id)::BIGINT AS n_customer_unique_id,
    (SELECT COUNT(*)::BIGINT FROM (
        SELECT c.customer_unique_id
        FROM olist_orders AS o
        JOIN olist_customers AS c ON c.customer_id = o.customer_id
        GROUP BY c.customer_unique_id
        HAVING COUNT(*) > 1
    ) AS t) AS n_unique_with_2plus_orders
FROM olist_customers
""")

,n_customer_id,n_customer_unique_id,n_unique_with_2plus_orders
0,99441,96096,2997


Pipeline повертає 99441 рядок, по одному на customer_id. frequency скрізь 1, бо в Olist customer_id створюється на кожне замовлення, а справжній клієнт це customer_unique_id (96096 штук, з них 2997 мають 2+ замовлення). Через це avg_order_value_30d_recent дорівнює monetary, а cancellation_rate буває тільки 0 або 1 (1 у 625 клієнтів). У 775 клієнтів preferred_category = '(none)', бо їхні замовлення без order_items. Залишив customer_id, як у шаблоні завдання.

## 3. Recursive CTE для referral chain

In [10]:
run("""
DROP TABLE IF EXISTS hw5_referrals CASCADE;
CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
""")

sql("""
SELECT referrer_customer_id, referred_customer_id
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;
""")

,referrer_customer_id,referred_customer_id
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130


In [11]:
sql("""
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*)::BIGINT AS n_customers,
    LEFT(MAX(path_text), 80) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
""")

,depth,n_customers,sample_path
0,1,99432,ffffe8b65bbe3087b653a978c870db99
1,2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140
2,3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd61
3,4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd61
4,5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd61
5,6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd61
6,7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd61
7,8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd61
8,9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd61
9,10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd61


Перевірка на циклі A → B → C → A (тимчасова таблиця без PK, старт з A).

In [12]:
CYCLE_SETUP = """
CREATE TEMP TABLE ref_cycle (referrer TEXT, referred TEXT) ON COMMIT DROP;
INSERT INTO ref_cycle VALUES ('A', 'B'), ('B', 'C'), ('C', 'A');
"""

def cycle_query(path_guard, depth_limit):
    guard = 'AND NOT r.referred = ANY(t.path_ids)' if path_guard else ''
    return f"""
WITH RECURSIVE t AS (
    SELECT 'A'::TEXT AS customer_id, 1 AS depth, ARRAY['A']::TEXT[] AS path_ids
    UNION ALL
    SELECT r.referred, t.depth + 1, t.path_ids || r.referred
    FROM ref_cycle AS r
    JOIN t ON t.customer_id = r.referrer
    WHERE t.depth < {depth_limit} {guard}
)
SELECT
    COUNT(*)::BIGINT AS n_rows,
    MAX(depth) AS max_depth,
    (SELECT array_to_string(path_ids[1:8], ' -> ') FROM t ORDER BY depth DESC LIMIT 1) AS longest_path_start
FROM t
"""

rows = []
with engine.begin() as conn:
    conn.execute(text(CYCLE_SETUP))
    for name, pg_, dl in [('path_ids + depth < 50', True, 50), ('тільки depth < 50', False, 50)]:
        r = conn.execute(text(cycle_query(pg_, dl))).mappings().one()
        rows.append({'захист': name, **r})
pd.DataFrame(rows)

,захист,n_rows,max_depth,longest_path_start
0,path_ids + depth < 50,3,3,A -> B -> C
1,тільки depth < 50,50,50,A -> B -> C -> A -> B -> C -> A -> B


У hw5_referrals 9 пар, це один ланцюжок з 10 клієнтів. Решта 99432 клієнти не мають referrer і йдуть як корені з depth 1, ланцюжок дає по одному клієнту на depth 2–10. На циклі A → B → C → A перевірка path_ids зупиняє рекурсію на depth 3. Якщо залишити тільки depth guard, запит ходить по колу до depth 50 і повертає 50 рядків замість 3.

## 4. Materialized View з CONCURRENT REFRESH

In [13]:
run("""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;
CREATE MATERIALIZED VIEW mv_olist_customer_features AS
""" + FEATURES_CTE + FEATURES_SELECT + """;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);
""")
print('MV і UNIQUE INDEX створено')

MV і UNIQUE INDEX створено


In [14]:
sql("""
SELECT COUNT(*)::BIGINT AS n_rows
FROM mv_olist_customer_features;
""")

,n_rows
0,99441


In [15]:
sql("""
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
""")

,customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66


In [16]:
import time
t0 = time.perf_counter()
run('REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;')
t_conc = time.perf_counter() - t0
t0 = time.perf_counter()
run('REFRESH MATERIALIZED VIEW mv_olist_customer_features;')
t_plain = time.perf_counter() - t0
print(f'REFRESH CONCURRENTLY: {t_conc:.2f} с')
print(f'REFRESH звичайний: {t_plain:.2f} с')

REFRESH CONCURRENTLY: 2.33 с
REFRESH звичайний: 1.31 с


In [17]:
sql("""
SELECT
    (SELECT COUNT(*) FROM mv_olist_customer_features)::BIGINT AS n_mv,
    (SELECT COUNT(*) FROM (""" + FEATURES_CTE + FEATURES_SELECT + """) AS q)::BIGINT AS n_pipeline,
    (SELECT COUNT(*) FROM (
        (""" + FEATURES_CTE + FEATURES_SELECT + """)
        EXCEPT
        SELECT * FROM mv_olist_customer_features
    ) AS d)::BIGINT AS n_diff
""")

,n_mv,n_pipeline,n_diff
0,99441,99441,0


MV має 99441 рядок, як і pipeline, EXCEPT між ними дає 0 відмінностей. REFRESH CONCURRENTLY спрацював з UNIQUE INDEX на customer_id. Він повільніший за звичайний (2.33 с проти 1.31 с), бо рахує новий результат окремо і порівнює зі старим, зате не блокує SELECT з MV під час оновлення.

## 5. EXPLAIN ANALYZE + Reflection

### 5.1. EXPLAIN на повному CTE-pipeline

In [18]:
explain(PIPELINE_TOP20)

Limit  (cost=116352.34..116352.39 rows=20 width=150) (actual time=1466.397..1467.765 rows=20 loops=1)
  Buffers: shared hit=111711, temp read=8395 written=8544
  CTE stage1_orders_total
    ->  GroupAggregate  (cost=0.83..27070.14 rows=99441 width=103) (actual time=0.041..189.388 rows=99441 loops=1)
          Group Key: o_1.order_id
          Buffers: shared hit=103453
          ->  Merge Left Join  (cost=0.83..24485.05 rows=112650 width=95) (actual time=0.023..104.508 rows=113425 loops=1)
                Merge Cond: (o_1.order_id = oi_1.order_id)
                Buffers: shared hit=103453
                ->  Index Scan using olist_orders_pkey on olist_orders o_1  (cost=0.42..12564.03 rows=99441 width=83) (actual time=0.011..45.157 rows=99441 loops=1)
                      Buffers: shared hit=100347
                ->  Index Scan using olist_order_items_pkey on olist_order_items oi_1  (cost=0.42..10264.29 rows=112650 width=45) (actual time=0.007..18.557 rows=112650 loops=1)
           

### 5.2. EXPLAIN на lookup із MV

In [19]:
explain("""
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id
""")

Nested Loop  (cost=0.83..8.96 rows=1 width=158) (actual time=0.053..0.054 rows=1 loops=1)
  Buffers: shared hit=5 read=3
  ->  Limit  (cost=0.42..0.53 rows=1 width=33) (actual time=0.030..0.030 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..11127.96 rows=99441 width=33) (actual time=0.029..0.029 rows=1 loops=1)
              Heap Fetches: 1
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=158) (actual time=0.019..0.019 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=1 read=3
Planning Time: 0.141 ms
Execution Time: 0.070 ms


### 5.3. Top-N з MV: без індексу на monetary і з ним

In [20]:
TOP10_MV = """
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10
"""
run('DROP INDEX IF EXISTS idx_mv_olist_customer_features_monetary;')
explain(TOP10_MV)

Limit  (cost=4449.01..4450.16 rows=10 width=158) (actual time=11.588..12.627 rows=10 loops=1)
  Buffers: shared hit=1636
  ->  Gather Merge  (cost=4449.01..11175.94 rows=58495 width=158) (actual time=11.587..12.623 rows=10 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=1636
        ->  Sort  (cost=3449.00..3595.24 rows=58495 width=158) (actual time=10.062..10.064 rows=8 loops=2)
              Sort Key: monetary DESC NULLS LAST
              Sort Method: top-N heapsort  Memory: 26kB
              Buffers: shared hit=1636
              Worker 0:  Sort Method: top-N heapsort  Memory: 27kB
              ->  Parallel Seq Scan on mv_olist_customer_features  (cost=0.00..2184.95 rows=58495 width=158) (actual time=0.007..4.541 rows=49720 loops=2)
                    Buffers: shared hit=1600
Planning Time: 0.052 ms
Execution Time: 12.645 ms


In [21]:
run("""
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);
ANALYZE mv_olist_customer_features;
""")
explain(TOP10_MV)

Limit  (cost=4449.01..4450.16 rows=10 width=92) (actual time=12.447..13.347 rows=10 loops=1)
  Buffers: shared hit=1636
  ->  Gather Merge  (cost=4449.01..11175.94 rows=58495 width=92) (actual time=12.445..13.342 rows=10 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=1636
        ->  Sort  (cost=3449.00..3595.24 rows=58495 width=92) (actual time=10.673..10.674 rows=10 loops=2)
              Sort Key: monetary DESC NULLS LAST
              Sort Method: top-N heapsort  Memory: 26kB
              Buffers: shared hit=1636
              Worker 0:  Sort Method: top-N heapsort  Memory: 27kB
              ->  Parallel Seq Scan on mv_olist_customer_features  (cost=0.00..2184.95 rows=58495 width=92) (actual time=0.009..4.744 rows=49720 loops=2)
                    Buffers: shared hit=1600
Planning:
  Buffers: shared hit=50 read=1
Planning Time: 0.171 ms
Execution Time: 13.368 ms


Індекс з шаблону не використався: у запиті `DESC NULLS LAST`, а індекс `DESC` за замовчуванням зберігає NULLS FIRST. Додав індекс з тим самим порядком.

In [22]:
run("""
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary_nl
    ON mv_olist_customer_features (monetary DESC NULLS LAST);
ANALYZE mv_olist_customer_features;
""")
explain(TOP10_MV)

Limit  (cost=0.42..1.32 rows=10 width=92) (actual time=0.017..0.026 rows=10 loops=1)
  Buffers: shared hit=10 read=3
  ->  Index Scan using idx_mv_olist_customer_features_monetary_nl on mv_olist_customer_features  (cost=0.42..8995.40 rows=99441 width=92) (actual time=0.015..0.024 rows=10 loops=1)
        Buffers: shared hit=10 read=3
Planning:
  Buffers: shared hit=60 read=1
Planning Time: 0.219 ms
Execution Time: 0.037 ms


### 5.4. Час окремих features (медіана з 5 запусків, Execution Time з EXPLAIN ANALYZE)

In [23]:
FULL_PIPELINE = FEATURES_CTE + FEATURES_SELECT
FEATURE_QUERIES = {
    'stage1 order_total': (FEATURES_CTE + 'SELECT * FROM stage1_orders_total', None),
    'RFM (stage1-2)': (FEATURES_CTE + 'SELECT * FROM stage2_rfm', None),
    'cohort_quarter (stage1-3)': (FEATURES_CTE + 'SELECT * FROM stage3_cohort', None),
    'avg_order_value_30d (stage1, 4)': (FEATURES_CTE + 'SELECT * FROM stage4_rolling_aov WHERE rn_recent = 1', None),
    'cancellation_rate (stage1, 5)': (FEATURES_CTE + 'SELECT * FROM stage5_cancel_rate', None),
    'preferred_category (stage6-7)': (FEATURES_CTE + 'SELECT * FROM stage7_preferred_category', None),
    'повний pipeline, всі клієнти': (FULL_PIPELINE, None),
    'повний pipeline, top-20': (PIPELINE_TOP20, None),
    'MV lookup за customer_id': ("SELECT * FROM mv_olist_customer_features WHERE customer_id = (SELECT MIN(customer_id) FROM olist_customers)", None),
    'MV top-10 за monetary': (TOP10_MV, None),
}
timing = pd.DataFrame(
    [{'запит': k, 'час, мс': exec_ms(q, setup=st)} for k, (q, st) in FEATURE_QUERIES.items()]
)
timing

,запит,"час, мс"
0,stage1 order_total,239.37
1,RFM (stage1-2),397.76
2,cohort_quarter (stage1-3),432.75
3,"avg_order_value_30d (stage1, 4)",480.37
4,"cancellation_rate (stage1, 5)",334.08
5,preferred_category (stage6-7),311.15
6,"повний pipeline, всі клієнти",1383.58
7,"повний pipeline, top-20",1410.92
8,MV lookup за customer_id,0.02
9,MV top-10 за monetary,0.01


Повний pipeline виконується за 1471 мс. Lookup з MV по customer_id іде через Index Scan по unique index, 0.07 мс. Top-10 з MV без відповідного індексу робить Parallel Seq Scan і top-N heapsort, 12.6 мс, з індексом monetary DESC NULLS LAST це Index Scan, 0.04 мс. З окремих features найдовше рахується avg_order_value_30d, 480 мс, з них 239 мс займає stage1.

### Reflection

Найдорожчий feature за EXPLAIN ANALYZE це avg_order_value_30d. Окремо він рахується 480 мс, з них 239 мс це stage1 з order_total. У плані під ним Sort по customer_id, order_ts DESC, order_id DESC з external merge на диск 9448kB, далі два WindowAgg та Incremental Sort. Тобто дорогі саме window functions і сортування для них. Схоже виглядає preferred_category: Sort з external merge і WindowAgg для ROW_NUMBER. RFM і cancellation_rate рахуються через HashAggregate, який при work_mem 4MB розбивається на 21 batch і пише на диск близько 7MB. stage1 матеріалізується як CTE один раз, бо його використовують три stage.

Кешувати через MV варто весь набір features, а не один. Повний pipeline займає близько 1.4 с, lookup з MV через Index Scan 0.07 мс, top-10 з індексом 0.04 мс. Дані Olist історичні і не змінюються, тому тут вистачить on-demand refresh після завантаження нових даних. Для живого магазину обрав би щодобовий REFRESH CONCURRENTLY вночі: recency_days рахується в днях, погодинне оновлення нічого не додає.

Звичайний VIEW кращий, коли потрібні завжди актуальні дані і запит дешевий. Наприклад, поточний статус замовлення з фільтром по індексу. Також VIEW має сенс, якщо запит викликається рідко і зберігати результат немає сенсу.

Recursive CTE завершується, коли recursive part не повертає нових рядків. У дереві це відбувається на листках. При циклі A → B → A новий рядок з'являється завжди, і без захисту запит не зупиниться. Тому потрібні дві перевірки: NOT next_id = ANY(path_ids) не пускає у вже пройдений вузол, а depth < 50 обмежує глибину. На тесті з циклом без path_ids вийшло 50 рядків замість 3. У PostgreSQL 14+ можна ще використати CYCLE-clause.

Для цього pipeline головний trade-off це freshness проти recompute-cost. Storage на 99441 рядок невеликий, а перерахунок 1.4 с на кожен запит дорогий.